## CKA estimator and sample-convention ablation

The paper's statistic is the biased linear CKA estimator on the time-pooled `[C, F]` representation (channels as samples, frequency bins as features). Its *Centered Kernel Alignment* section reports an ablation that recomputes the MUSE profile with the unbiased HSIC estimator on the same pooled representation (and, as a further check, with time frames rather than channels as samples) on a reduced grid of 150 utterances x 9 SNR levels x 2 DEMAND noises with non-pooled activations: the unbiased estimator reproduces the biased one nearly exactly, at a per-layer slope correlation of 0.98 and with the same peak layer. This chapter reproduces that comparison from the shipped per-layer slopes and demonstrates the estimators in `se_probe.cka`.

Runtime: under 30 seconds on CPU; the gated live cell adds about a minute.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
TABLES = REPO / "results_tables"
print(f"Using data from: {DATA_DIR}")
print(f"Paper tables from: {TABLES}")

## Per-layer slopes under the four variants

`results_tables/ablation/hsic_frames_slopes.csv` holds the per-layer OLS slope of CKA against SNR on the reduced grid for the four variants: `pooled_linear` (the paper's statistic), `hsic_unbiased` (same `[C, F]` representation, unbiased HSIC estimator), `frames_TFC` (`[T, C*F]`, frames as samples) and `frames_TC` (`[T, C]`, the Demucs-style convention). Only the estimator swap leaves the profile intact; changing what counts as a sample changes which invariance is measured and gives a different profile, which is why profiles are comparable only within a fixed sample-axis convention.

In [ ]:
import numpy as np

from se_probe.profiles import profile_agreement

ab = pd.read_csv(TABLES / "ablation/hsic_frames_slopes.csv")
VARIANTS = ["pooled_linear", "hsic_unbiased", "frames_TFC", "frames_TC"]
rows = []
for v in VARIANTS:
    ag = profile_agreement(ab.pooled_linear, ab[v])
    rows.append({"variant": v, "pearson_r_vs_pooled": ag["pearson_r"], "spearman_rho_vs_pooled": ag["spearman_rho"],
                 "peak_layer": ab.layer_short[ab[v].idxmax()], "peak_beta": ab[v].max()})
agree = pd.DataFrame(rows).set_index("variant")
display(agree.round(4))
r_hsic = agree.loc["hsic_unbiased", "pearson_r_vs_pooled"]
print(f"unbiased HSIC vs biased: r = {r_hsic:.3f} (paper 0.98), peak layer {agree.loc['hsic_unbiased', 'peak_layer']} "
      f"vs {agree.loc['pooled_linear', 'peak_layer']}")
assert abs(r_hsic - 0.98) < 5e-3
assert agree.loc["hsic_unbiased", "peak_layer"] == agree.loc["pooled_linear", "peak_layer"]
assert agree.loc["frames_TFC", "pearson_r_vs_pooled"] < 0 and agree.loc["frames_TC", "pearson_r_vs_pooled"] < 0

In [ ]:
import matplotlib.pyplot as plt

from se_probe.layers import layout
from se_probe.plotting import OKABE_ITO, depth_axis

colors = {"pooled_linear": OKABE_ITO["blue"], "hsic_unbiased": OKABE_ITO["vermillion"],
          "frames_TFC": OKABE_ITO["green"], "frames_TC": OKABE_ITO["purple"]}
markers = {"pooled_linear": "o", "hsic_unbiased": "s", "frames_TFC": "^", "frames_TC": "d"}
labels = {"pooled_linear": "pooled linear (paper)", "hsic_unbiased": "unbiased HSIC",
          "frames_TFC": "frames as samples [T, C*F]", "frames_TC": "[T, C] convention"}
fig, ax = plt.subplots(figsize=(8.5, 3.6))
depth_axis(ax, layout("muse"))
x = np.arange(len(ab))
for v in VARIANTS:
    ax.plot(x, ab[v], marker=markers[v], color=colors[v], lw=1.9 if v == "pooled_linear" else 1.2,
            label=f"{labels[v]} (r = {agree.loc[v, 'pearson_r_vs_pooled']:.2f})")
ax.set_ylabel(r"$\beta_\ell$ (CKA/dB)")
ax.set_title("MUSE sensitivity profile under estimator and sample-convention changes (150 utts x 9 SNR x 2 noises)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout();

## The estimators, live on synthetic activations

`cka_variants` builds the three representations from one non-pooled first-segment activation of shape `(C, T, F)` and returns all four values. The biased estimator is exact only as the number of samples grows; the unbiased HSIC estimator removes the finite-sample bias, which is visible below as the two agreeing more closely as `n = C` increases. `linear_cka_biased` on the pooled matrix is the same number as `cka()` from chapter 01.

In [ ]:
import importlib

import torch

cka_mod = importlib.import_module("se_probe.cka")

rng = torch.Generator().manual_seed(0)
rows = []
for C in (8, 16, 32, 48, 96):
    clean = torch.randn(C, 20, 64, generator=rng)
    degraded = clean + 0.6 * torch.randn(C, 20, 64, generator=rng)
    v = cka_mod.cka_variants(clean, degraded)
    rows.append({"n_samples (C)": C, **{k: round(val, 4) for k, val in v.items()}})
display(pd.DataFrame(rows).set_index("n_samples (C)"))

pooled_c, pooled_d = clean.mean(dim=1), degraded.mean(dim=1)
b = cka_mod.linear_cka_biased(pooled_c, pooled_d)
ref = cka_mod.cka(pooled_c[None], pooled_d[None])[0]
print(f"linear_cka_biased = {b:.6f}, cka() on the pooled matrix = {ref:.6f}")
assert abs(b - ref) < 1e-6

## Optional: one real activation

Set `SE_PROBE_RUN_INFERENCE=1` (after `python scripts/setup.py`) to extract a non-pooled activation of the first probed MUSE layer on a synthetic clip and print the four variants.

In [ ]:
import os

if os.environ.get("SE_PROBE_RUN_INFERENCE") == "1":
    from se_probe.activation_extraction import load_muse_activation_extractor

    extractor = load_muse_activation_extractor(device=DEVICE, with_pooling=False)
    g = np.random.default_rng(0)
    sr = 16000
    clean = (g.standard_normal(sr * 2) * 0.05).astype("float32")
    noise = (g.standard_normal(sr * 2) * 0.05).astype("float32")
    noisy = clean + noise
    act_c, _ = extractor(clean)
    act_n, _ = extractor(noisy)
    layer = next(iter(act_c))
    print(layer, {k: round(v, 4) for k, v in cka_mod.cka_variants(act_c[layer].float(), act_n[layer].float()).items()})
else:
    print("SE_PROBE_RUN_INFERENCE not set — skipping live extraction. Set it to '1' to enable.")